<a href="https://colab.research.google.com/github/kocakcan/ml_foundations/blob/main/intro_to_bert.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
model = BertModel.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
named_params = list(model.named_parameters())
print("The BERT model has {:} different named parameters.\n".format(len(named_params)))

print("=== Embedding Layer ===\n")
for p in named_params[0:5]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print("\n=== First Encoder ===\n")
for p in named_params[5:21]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

print("\n=== Output Layer ===\n")
for p in named_params[-2:]:
  print("{:<55} {:>12}".format(p[0], str(tuple(p[1].size()))))

The BERT model has 199 different named parameters.

=== Embedding Layer ===

embeddings.word_embeddings.weight                       (30522, 768)
embeddings.position_embeddings.weight                     (512, 768)
embeddings.token_type_embeddings.weight                     (2, 768)
embeddings.LayerNorm.weight                                   (768,)
embeddings.LayerNorm.bias                                     (768,)

=== First Encoder ===

encoder.layer.0.attention.self.query.weight               (768, 768)
encoder.layer.0.attention.self.query.bias                     (768,)
encoder.layer.0.attention.self.key.weight                 (768, 768)
encoder.layer.0.attention.self.key.bias                       (768,)
encoder.layer.0.attention.self.value.weight               (768, 768)
encoder.layer.0.attention.self.value.bias                     (768,)
encoder.layer.0.attention.output.dense.weight             (768, 768)
encoder.layer.0.attention.output.dense.bias                   (768,)
en

In [4]:
# The pooler is a separate linear and tanh activated layer that acts on the [CLS] token's representation
# This pooled_output is often used as a representation for the entire sentence.

In [5]:
# load the bert-base uncased tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [6]:
tokenizer.encode("Can finds NLP to be highly intriguing.")

[101, 2064, 4858, 17953, 2361, 2000, 2022, 3811, 23824, 1012, 102]

In [7]:
# run tokens through the model

#1 Turn tokens_with_unknown_words into a tensor (will be size (8,))
#2 Unsquueze a first dimension to simulate batches. Resulting shape is (1, 8)
response = model(torch.tensor(tokenizer.encode("Can finds NLP to be highly intriguing.")).unsqueeze(0))

In [8]:
response

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[-0.2760, -0.2602, -0.0949,  ..., -0.3765,  0.3917,  0.4869],
         [ 0.0272,  0.0338,  0.5508,  ..., -0.2681,  0.4822,  0.3296],
         [-0.9126, -0.0358,  0.3538,  ..., -0.7194,  0.4401, -0.2475],
         ...,
         [ 0.2202,  0.1291,  0.3081,  ..., -0.3098,  0.1109, -0.0429],
         [-0.0573, -0.2961, -0.3703,  ...,  0.4762, -0.0134, -0.4318],
         [ 0.5397, -0.0829, -0.3930,  ...,  0.2242, -0.6757, -0.3840]]],
       grad_fn=<NativeLayerNormBackward0>), pooler_output=tensor([[-7.9536e-01, -2.2278e-01, -6.4746e-01,  6.0603e-01,  4.7414e-01,
          2.2248e-02,  7.3109e-01,  2.7776e-01, -4.7601e-01, -9.9996e-01,
         -3.2936e-02,  3.5850e-01,  9.6317e-01,  1.6054e-01,  8.0588e-01,
         -4.7449e-01,  1.1769e-01, -4.9254e-01,  1.9543e-01, -3.9526e-01,
          5.4644e-01,  9.9966e-01,  2.2856e-01,  2.3224e-01,  2.7254e-01,
          7.4438e-01, -5.5636e-01,  8.5589e-01,  9.3338e-01,  7.630

In [9]:
response.last_hidden_state

tensor([[[-0.2760, -0.2602, -0.0949,  ..., -0.3765,  0.3917,  0.4869],
         [ 0.0272,  0.0338,  0.5508,  ..., -0.2681,  0.4822,  0.3296],
         [-0.9126, -0.0358,  0.3538,  ..., -0.7194,  0.4401, -0.2475],
         ...,
         [ 0.2202,  0.1291,  0.3081,  ..., -0.3098,  0.1109, -0.0429],
         [-0.0573, -0.2961, -0.3703,  ...,  0.4762, -0.0134, -0.4318],
         [ 0.5397, -0.0829, -0.3930,  ...,  0.2242, -0.6757, -0.3840]]],
       grad_fn=<NativeLayerNormBackward0>)

In [10]:
response.pooler_output.shape

torch.Size([1, 768])

In [11]:
model.pooler

BertPooler(
  (dense): Linear(in_features=768, out_features=768, bias=True)
  (activation): Tanh()
)

In [12]:
# grab the final encoder's representation of the CLS token
CLS_embedding = response.last_hidden_state[:, 0, :].unsqueeze(0)
CLS_embedding.shape

torch.Size([1, 1, 768])

In [13]:
model.pooler(CLS_embedding).shape

torch.Size([1, 768])

In [14]:
# Running the embedding for CLS through the pooler gives the same output as the `pooler_output`
(model.pooler(CLS_embedding) == response.pooler_output).all()

tensor(True)

In [15]:
total_params = 0
for p in model.parameters():
  if len(p.shape) == 2:
    total_params += p.shape[0] * p.shape[1]
print(f"Total parameters: {total_params:,}")

Total parameters: 109,360,128


In [16]:
"Can" in tokenizer.vocab

False

In [17]:
# BERT's tokenizer is great at handling tokens that are OOV (out of vocabulary) by breaking them up into smaller chunks of known tokens

In [18]:
tokenizer.encode("I love my pet Python.")

[101, 1045, 2293, 2026, 9004, 18750, 1012, 102]

In [19]:
tokenizer.encode("I love coding in Python.")

[101, 1045, 2293, 16861, 1999, 18750, 1012, 102]

In [20]:
# The token `python` will end up with a vector representation from each sentence via BERT.
# What's interesting is that the vector representation `python` will be different for each sentence
# because of the surrounding words in the sentence

In [21]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
print(f"Length of BERT base vocabulary: {len(tokenizer.vocab)}")

Length of BERT base vocabulary: 30522


In [22]:
text = "A simple sentence!"
# get token ids per BERT-base's vocabulary
tokens = tokenizer.encode(text)
print(tokens)

[101, 1037, 3722, 6251, 999, 102]


In [23]:
# decode will re-construct the sentence with the added [CLS] and [SEP] tokens
tokenizer.decode(tokens)

'[CLS] a simple sentence! [SEP]'

In [24]:
text = "My friend told me about this class and I love it so far! She was right."

tokens = tokenizer.encode(text)
print(tokens)

[101, 2026, 2767, 2409, 2033, 2055, 2023, 2465, 1998, 1045, 2293, 2009, 2061, 2521, 999, 2016, 2001, 2157, 1012, 102]


In [25]:
# A nicer printout of token ids and token strings
for t in tokens:
  print(f"Token: {t}, subword: {tokenizer.decode([t])}")

Token: 101, subword: [CLS]
Token: 2026, subword: my
Token: 2767, subword: friend
Token: 2409, subword: told
Token: 2033, subword: me
Token: 2055, subword: about
Token: 2023, subword: this
Token: 2465, subword: class
Token: 1998, subword: and
Token: 1045, subword: i
Token: 2293, subword: love
Token: 2009, subword: it
Token: 2061, subword: so
Token: 2521, subword: far
Token: 999, subword: !
Token: 2016, subword: she
Token: 2001, subword: was
Token: 2157, subword: right
Token: 1012, subword: .
Token: 102, subword: [SEP]


In [26]:
texts = ["I love coding in Python", "I love my pet Python"]
for text in texts:
  tokens = tokenizer.encode(text)
  for t in tokens:
    print(f"Token: {t}, subword: {tokenizer.decode([t])}")
  print()

Token: 101, subword: [CLS]
Token: 1045, subword: i
Token: 2293, subword: love
Token: 16861, subword: coding
Token: 1999, subword: in
Token: 18750, subword: python
Token: 102, subword: [SEP]

Token: 101, subword: [CLS]
Token: 1045, subword: i
Token: 2293, subword: love
Token: 2026, subword: my
Token: 9004, subword: pet
Token: 18750, subword: python
Token: 102, subword: [SEP]



In [27]:
"Can" in tokenizer.vocab

False

In [28]:
text_with_unknown_words = "Medet loves a beautiful day"
tokens_with_unknown_words = tokenizer.encode(text_with_unknown_words)

# We see our sub words in action:
for t in tokens_with_unknown_words:
  print(f"Token: {t}, subword: {tokenizer.decode([t])}")

Token: 101, subword: [CLS]
Token: 19960, subword: med
Token: 3388, subword: ##et
Token: 7459, subword: loves
Token: 1037, subword: a
Token: 3376, subword: beautiful
Token: 2154, subword: day
Token: 102, subword: [SEP]


In [29]:
tokenizer.encode("Medet")

[101, 19960, 3388, 102]

In [30]:
text_with_unknown_words = "Can is our instructor for this awesomesauce class"
tokens_with_unknown_words = tokenizer.encode(text_with_unknown_words)

for t in tokens_with_unknown_words:
  print(f"Token: {t}, subword: {tokenizer.decode([t])}")

Token: 101, subword: [CLS]
Token: 2064, subword: can
Token: 2003, subword: is
Token: 2256, subword: our
Token: 9450, subword: instructor
Token: 2005, subword: for
Token: 2023, subword: this
Token: 12476, subword: awesome
Token: 23823, subword: ##sau
Token: 3401, subword: ##ce
Token: 2465, subword: class
Token: 102, subword: [SEP]


In [31]:
text = "My friend told me about this class and I love it so far! She was right."

# encode_plus gives us token ids, attention mask and segment ids (A vs B). Useful for training time
# tokens = tokenizer.encode_plus(text)
# print(tokens)

In [32]:
# python is the 6th token (don't forget the [CLS] token!)
python_pet = tokenizer.encode("I love my pet python")

# python is the 6th token (don't forget the [CLS] token!)
python_language = tokenizer.encode("I love coding in python")

In [33]:
# contextful embedding of `python` in "I love my pet python"
python_pet_embedding = model(torch.tensor(python_pet).unsqueeze(0))[0][:, 5, :].detach().numpy()

# contextful embedding of `python` in "I love coding in python"
python_language_embedding = model(torch.tensor(python_language).unsqueeze(0))[0][:, 5, :].detach().numpy()

# contextful embedding of `snake` in "snake"
snake_alone_embedding = model(torch.tensor(tokenizer.encode("snake")).unsqueeze(0))[0][:, 1, :].detach().numpy()

# contextful embedding of `snake` in "programming"
programming_alone_embedding = model(torch.tensor(tokenizer.encode("programming")).unsqueeze(0))[0][:, 1, :].detach().numpy()


In [34]:
python_pet_embedding.shape

(1, 768)

In [35]:
python_language_embedding.shape

(1, 768)

In [36]:
cosine_similarity(python_language_embedding, snake_alone_embedding)

array([[0.5843479]], dtype=float32)

In [37]:
cosine_similarity(python_pet_embedding, snake_alone_embedding)

array([[0.6928656]], dtype=float32)

In [38]:
cosine_similarity(python_pet_embedding, programming_alone_embedding)

array([[0.4986436]], dtype=float32)

In [39]:
cosine_similarity(python_language_embedding, programming_alone_embedding)

array([[0.5614742]], dtype=float32)

BERT applies three separate types of embeddings to tokenized sentences:
1.   Token Embeddings
::*   Represents context-less meaning of each token
*   A lookup of 30,522 possible vectors (for BERT-base)
*   This is learnable during training
2.   Segment Embeddings
*   Distinguishes between multiple inputs (for Q/A for example)
*   A lookup of 2 possible vectors (one for sentence A, one for sentence B)
*   This is not learnable
3.   Position Embeddings
*   Used to represent the token's position in the sentence
*   This is not learnable


In [40]:
"""
word_embeddings == context-free word embeddings
position_embeddings == encodes word position
token_type_embeddings == 0 or 1. Used to lookup the segment embedding
"""

'\nword_embeddings == context-free word embeddings\nposition_embeddings == encodes word position\ntoken_type_embeddings == 0 or 1. Used to lookup the segment embedding\n'

In [41]:
model.embeddings

BertEmbeddings(
  (word_embeddings): Embedding(30522, 768, padding_idx=0)
  (position_embeddings): Embedding(512, 768)
  (token_type_embeddings): Embedding(2, 768)
  (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
  (dropout): Dropout(p=0.1, inplace=False)
)

In [42]:
example_phrase = "I am Can"

# return tensors='pt' converts to pytorch automatically
tokenizer.encode(example_phrase, return_tensors="pt")

tensor([[ 101, 1045, 2572, 2064,  102]])

In [43]:
# context-less embedding of each token in our sentence
model.embeddings.word_embeddings(tokenizer.encode(example_phrase, return_tensors="pt"))

tensor([[[ 0.0136, -0.0265, -0.0235,  ...,  0.0087,  0.0071,  0.0151],
         [-0.0211,  0.0059, -0.0179,  ...,  0.0163,  0.0122,  0.0073],
         [-0.0437, -0.0150,  0.0029,  ..., -0.0282,  0.0474, -0.0448],
         [ 0.0546, -0.0655,  0.0345,  ..., -0.0347, -0.0261, -0.0851],
         [-0.0145, -0.0100,  0.0060,  ..., -0.0250,  0.0046, -0.0015]]],
       grad_fn=<EmbeddingBackward0>)

In [44]:
# Note the first and last row are the same because the are the
# [CLS] and [SEP] reserved tokens. They are the same without context for every input
model.embeddings.word_embeddings(tokenizer.encode("I am Medet", return_tensors="pt"))

tensor([[[ 0.0136, -0.0265, -0.0235,  ...,  0.0087,  0.0071,  0.0151],
         [-0.0211,  0.0059, -0.0179,  ...,  0.0163,  0.0122,  0.0073],
         [-0.0437, -0.0150,  0.0029,  ..., -0.0282,  0.0474, -0.0448],
         [-0.0078, -0.1354, -0.0754,  ..., -0.0052, -0.0232, -0.0651],
         [ 0.0102, -0.0576,  0.0400,  ...,  0.0170, -0.0408, -0.0005],
         [-0.0145, -0.0100,  0.0060,  ..., -0.0250,  0.0046, -0.0015]]],
       grad_fn=<EmbeddingBackward0>)

In [45]:
# 512 embeddings, one for each position in a max 512 input sequence
model.embeddings.position_embeddings

Embedding(512, 768)

In [46]:
torch.LongTensor(range(5))

tensor([0, 1, 2, 3, 4])

In [47]:
# positional embeddings for our example_phrase
model.embeddings.position_embeddings(torch.LongTensor(range(5)))

tensor([[ 1.7505e-02, -2.5631e-02, -3.6642e-02,  ...,  3.3437e-05,
          6.8312e-04,  1.5441e-02],
        [ 7.7580e-03,  2.2613e-03, -1.9444e-02,  ...,  2.8910e-02,
          2.9753e-02, -5.3247e-03],
        [-1.1287e-02, -1.9644e-03, -1.1573e-02,  ...,  1.4908e-02,
          1.8741e-02, -7.3140e-03],
        [-4.1949e-03, -1.1852e-02, -2.1180e-02,  ...,  2.2455e-02,
          5.2826e-03, -1.9723e-03],
        [-5.6087e-03, -1.0445e-02, -7.2288e-03,  ...,  2.0837e-02,
          3.5402e-03,  4.7708e-03]], grad_fn=<EmbeddingBackward0>)

In [48]:
# 2 embeddings. One for A and one for B
model.embeddings.token_type_embeddings

Embedding(2, 768)

In [49]:
# All tokens have the same embedding
model.embeddings.token_type_embeddings(torch.LongTensor([0] * 5))

tensor([[ 0.0004,  0.0110,  0.0037,  ..., -0.0066, -0.0034, -0.0086],
        [ 0.0004,  0.0110,  0.0037,  ..., -0.0066, -0.0034, -0.0086],
        [ 0.0004,  0.0110,  0.0037,  ..., -0.0066, -0.0034, -0.0086],
        [ 0.0004,  0.0110,  0.0037,  ..., -0.0066, -0.0034, -0.0086],
        [ 0.0004,  0.0110,  0.0037,  ..., -0.0066, -0.0034, -0.0086]],
       grad_fn=<EmbeddingBackward0>)

In [50]:
# Apply feed forward normalization layer
model.embeddings.LayerNorm(
    model.embeddings.word_embeddings(tokenizer.encode(example_phrase, return_tensors="pt")) + \
    model.embeddings.position_embeddings(torch.LongTensor(range(5))) + \
    model.embeddings.token_type_embeddings(torch.LongTensor([0] * 5))
)

tensor([[[ 1.6855e-01, -2.8577e-01, -3.2613e-01,  ..., -2.7571e-02,
           3.8253e-02,  1.6400e-01],
         [-3.4024e-04,  5.3974e-01, -2.8805e-01,  ...,  7.5731e-01,
           8.9008e-01,  1.6575e-01],
         [-6.3496e-01,  1.9748e-01,  2.5116e-01,  ..., -4.0819e-02,
           1.3468e+00, -6.9357e-01],
         [ 1.0899e+00, -8.8606e-01,  5.1840e-01,  ..., -1.0336e-01,
          -1.4621e-01, -1.3990e+00],
         [-3.6430e-01, -1.6172e-01,  9.0174e-02,  ..., -1.7849e-01,
           1.2818e-01, -4.5116e-02]]], grad_fn=<NativeLayerNormBackward0>)

In [51]:
# Et Voilà! The many embeddings of BERT become one embedding per token
model.embeddings(tokenizer.encode(example_phrase, return_tensors="pt"))

tensor([[[ 1.6855e-01, -2.8577e-01, -3.2613e-01,  ..., -2.7571e-02,
           3.8253e-02,  1.6400e-01],
         [-3.4026e-04,  5.3974e-01, -2.8805e-01,  ...,  7.5731e-01,
           8.9008e-01,  1.6575e-01],
         [-6.3496e-01,  1.9748e-01,  2.5116e-01,  ..., -4.0819e-02,
           1.3468e+00, -6.9357e-01],
         [ 1.0899e+00, -8.8606e-01,  5.1840e-01,  ..., -1.0336e-01,
          -1.4621e-01, -1.3990e+00],
         [-3.6430e-01, -1.6172e-01,  9.0174e-02,  ..., -1.7849e-01,
           1.2818e-01, -4.5116e-02]]], grad_fn=<NativeLayerNormBackward0>)

In [52]:
model.embeddings(tokenizer.encode(example_phrase, return_tensors="pt")).shape

torch.Size([1, 5, 768])

In [53]:
from transformers import BertForMaskedLM, pipeline

In [55]:
bert_lm = BertForMaskedLM.from_pretrained("bert-base-cased")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [56]:
bert_lm

BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_a

In [59]:
# Pipelines in transformers take in models/tokenizers and are easy way to perform several tasks

# We can perform an auto-encoder language model task
nlp = pipeline("fill-mask", model="bert-base-cased")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

BertForMaskedLM LOAD REPORT from: bert-base-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [63]:
type(nlp.model)

transformers.models.bert.modeling_bert.BertForMaskedLM

In [66]:
nlp.tokenizer

BertTokenizer(name_or_path='bert-base-cased', vocab_size=28996, model_max_length=512, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

In [67]:
print(type(nlp.model))

preds = nlp(f"If you don't {nlp.tokenizer.mask_token} at the sign, you will get a ticket.")
for p in preds:
  print(f"Token: {p['token_str']}. Score: {100 * p['score']:,.2f}%")

<class 'transformers.models.bert.modeling_bert.BertForMaskedLM'>
Token: stop. Score: 51.10%
Token: look. Score: 38.41%
Token: arrive. Score: 1.11%
Token: glance. Score: 1.05%
Token: turn. Score: 0.72%
